# Preschool Resource Allocation Index (PRAI)

This notebook demonstrates the reproducible MVP workflow for the Preschool Resource Allocation Engine. It uses the synthetic city-by-year dataset in `datasets/sample_preschool_data.csv`. The notebook contains no precomputed results: outputs are generated only when the cells are run.

The default score applies the theory-led equal-dimension weighting specified in `docs/Resource_Allocation_Index.md`. Entropy weighting is available in the module as a sensitivity-analysis option, not as the default research specification.

## 1. Setup and data loading

The project root is resolved from the notebook location so that the workflow does not depend on a user-specific absolute path.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from models.allocation import load_data

data_path = PROJECT_ROOT / 'datasets' / 'sample_preschool_data.csv'
raw_data = load_data(data_path)
raw_data.head()

## 2. Indicator preparation

The engine derives demand-relative and per-child indicators from the raw city-year fields. Population is used as a denominator for access and capacity measures; it is not scored as a resource by itself.

In [ ]:
from models.allocation import prepare_indicators

indicators = prepare_indicators(raw_data)
indicators.head()

## 3. Indicator normalisation and weighting

For this synthetic MVP dataset, the module applies sample-based min–max normalisation. Positive and negative indicators are handled in opposite directions, while capacity pressure is assessed against an explicit target range. Results are comparable only within the jointly normalised sample.

In [ ]:
from models.allocation import calculate_weights, normalize_indicators

normalised_indicators = normalize_indicators(indicators)
mvp_weights = calculate_weights(normalised_indicators, method='equal')

mvp_weights.to_frame()

## 4. PRAI calculation and result presentation

The score is reported on a 0–100 scale. The three operational labels shown here are a concise reporting aid; substantive interpretation should also inspect the component indicators and methodological limits documented in the PRAI design.

In [ ]:
from models.allocation import calculate_prai_score, evaluate_level

results = calculate_prai_score(raw_data, weight_method='equal')
results['allocation_level'] = evaluate_level(results['resource_allocation_score'])
results.sort_values(['year', 'resource_allocation_score'], ascending=[True, False]).head(10)

## Reproducibility note

Before applying this workflow to real data, replace sample-based normalisation bounds and the provisional capacity-pressure target range with documented policy or theoretical benchmarks. Record data sources, definitions, transformations, and sensitivity analyses with every substantive PRAI application.